# **Scripts Documentation**

The Scripts Documentation is dedicated to explaining what are the scrips and what each one of them are used for.

The scripts are part of TAGI's project, in the way that they need to be runned so that the robot can function.

This project has a total of four scripts:

> #### start.sh

``start.sh`` initializes the execution environment by exporting display and runtime variables before executing a synchronous setup sequence. It calls ``middleware.py`` (see middleware's documentation for more information), reset to flush the global Redis state and ``load_config.py`` to populate the parameter server, ensuring a deterministic starting state for all subsequent modules.

The script then triggers the asynchronous deployment of the robot’s multi-process architecture using the ``&`` operator. This concurrently spawns the Hardware Abstraction Layer (HAL) drivers, network services (HTTP/MJPEG), and autonomous behavior nodes, allowing each independent process to communicate via the shared middleware bus.

In [ ]:
 #! /bin/bash

# Set environment for X11 / Pi if needed
export XDG_RUNTIME_DIR=/run/user/1000
export DISPLAY=0:0

# Wait for UART0 to be ready
#UART_DEVICE="/dev/serial0"  # could be /dev/ttyAMA0 on some models
#echo "Waiting for UART device $UART_DEVICE to be ready..."

#while [ ! -e "$UART_DEVICE" ]; do
#    echo "$(date): UART device $UART_DEVICE not ready yet. Sleeping 2 seconds..."
#    sleep 2
#done

echo "Waiting for Redis..."
until redis-cli ping >/dev/null; do
    sleep 2
done
echo "Redis is ready!"

# Ensure hostnames
grep -q "127.0.0.1 elmo" /etc/hosts || echo "127.0.0.1 elmo" | sudo tee -a /etc/hosts
grep -q "127.0.0.1 elmo2" /etc/hosts || echo "127.0.0.1 elmo2" | sudo tee -a /etc/hosts

mkdir -p /home/idmind/elmo-v2/logs

cd /home/idmind/elmo-v2/src
source /home/idmind/elmo-v2/.venv/bin/activate
python middleware.py reset > /dev/null
python load_config.py > /dev/null

python driver_battery.py >> /home/idmind/elmo-v2/logs/driver_battery.log &
python driver_gpio.py >> /home/idmind/elmo-v2/logs/driver_gpio.log &
sudo -n /home/idmind/elmo-v2/.venv/bin/python driver_leds.py >> /home/idmind/elmo-v2/logs/driver_leds.log &
# python driver_microphone.py >> /home/idmind/elmo-v2/logs/driver_microphone.log &
python driver_pan_tilt.py >> /home/idmind/elmo-v2/logs/driver_pan_tilt.log &
python driver_power.py >> /home/idmind/elmo-v2/logs/driver_power.log &
python driver_speakers.py >> /home/idmind/elmo-v2/logs/driver_speakers.log &
# python driver_speech.py >> /home/idmind/elmo-v2/logs/driver_speech.log &
python driver_touch_sensors.py >> /home/idmind/elmo-v2/logs/driver_touch_sensors.log &

python http_server.py >> /home/idmind/elmo-v2/logs/http_server.log &
python robot_api.py >> /home/idmind/elmo-v2/logs/robot_api.log &
python touch_calibrator.py >> /home/idmind/elmo-v2/logs/touch_calibrator.log &
python mjpeg_server_2.py >> /home/idmind/elmo-v2/logs/mjpeg_server_2.log &
python motor_temperature_watchdog.py >> /home/idmind/elmo-v2/logs/motor_temperature_watchdog.log &

python behaviour_blush.py >> /home/idmind/elmo-v2/logs/behaviour_blush.log &
python behaviour_test_motors.py >> /home/idmind/elmo-v2/logs/behaviour_test_motors.log &
# python behaviour_wifi_connect.py >> /home/idmind/elmo-v2/logs/behaviour_wifi_connect.log &
(sleep 8; python behaviour_photographer.py) >> /home/idmind/elmo-v2/logs/behaviour_photographer.log &
(sleep 8; python behaviour_hello.py) >> /home/idmind/elmo-v2/logs/behaviour_hello.log &

(sleep 5; python mode_manager.py >> /home/idmind/elmo-v2/logs/mode_manager.log) &

sleep 2
exec > /tmp/kiosk.log 2>&1
/bin/bash /home/idmind/elmo-v2/scripts/start_webapp.sh &

> #### start_webapp.sh

The script initiates the frontend deployment phase by enforcing a synchronous dependency check on the local network stack. It utilizes a ``until`` polling loop with ``curl`` to block execution until the backend HTTP service at port 8000 is fully reachable, preventing the browser from loading a "Connection Refused" error page during the boot sequence.

Upon validation, the script launches Chromium in Kiosk Mode, stripping away window decorations and navigation UI to provide a dedicated full-screen interface. The inclusion of a ``$RANDOM`` seed in the URL string serves as a cache-busting mechanism, ensuring that the robot's onboard display bypasses stored data and fetches fresh assets from the local server during every initialization.

In [ ]:
#! /bin/bash

export DISPLAY=:0
export XDG_RUNTIME_DIR=/run/user/1000

# Wait for PipeWire and display to exist
until pw-cli info 0 &>/dev/null; do
  echo "Waiting for PipeWire..."
  sleep 2
done

until xset q &>/dev/null; do
  echo "Waiting for display..."
  sleep 2
done

# set default microphone to virtual noise cancelled using PipeWire native commands
MICROPHONE_NAME="alsa_input.usb-C-Media_Electronics_Inc._USB_PnP_Sound_Device-00.analog-mono"

# Wait for the microphone to be available and get its node ID
MICROPHONE_ID="30"
#until [ -n "$MICROPHONE_ID" ]; do
# MICROPHONE_ID=$(pw-dump Node | grep -B5 -A5 "$MICROPHONE_NAME" | grep '"id"' | head -1 | grep -oP ':\s*\K\d+')
#  [ -z "$MICROPHONE_ID" ] && sleep 1
#done

# Set as default source
pw-metadata -n default 0 default.audio.source "{ \"name\": \"$MICROPHONE_NAME\" }"

# Set volume to 40% (0.4 in PipeWire)
pw-cli set-param "$MICROPHONE_ID" Props '{ "volume": 0.4, "mute": false }'

# Wait for the server
until $(curl --output /dev/null --silent --head --fail http://localhost:8000); do
  sleep 1
done

CHROMIUM_CMD="/usr/bin/chromium"

# Start the webapp
$CHROMIUM_CMD \
  --disable-gpu \
  --use-fake-ui-for-media-stream \
  --password-store=basic \
  --kiosk \
  --app="http://localhost:8000?p=$RANDOM" &

> #### install.sh

``install.sh`` executes a system-level dependency provisioning sequence, beginning with a package repository synchronization and an automated OS upgrade. It establishes the Redis message broker as the foundational middleware layer by importing official GPG keys, installing the server-side daemon, and deploying the Python client required for the robot's inter-process communication (IPC) bus.

The script then installs the Hardware Abstraction Layer (HAL) drivers and media utilities, specifically the Neopixel libraries for low-level control of the LED matrix and the ``gTTS/mpg123`` stack for synthetic speech generation. It also deploys Chromium, which functions as the primary Human-Machine Interface (HMI) for rendering the robot's web-based facial expressions.

In [ ]:
#! /usr/bin/bash

sudo apt update
sudo apt upgrade -y

# Redis -  In this Raspberry, REDIS must be compiled from source due to ARM architecture
#curl -fsSL https://packages.redis.io/gpg | sudo gpg --dearmor -o /usr/share/keyrings/redis-archive-keyring.gpg
#echo "deb [signed-by=/usr/share/keyrings/redis-archive-keyring.gpg] https://packages.redis.io/deb $(lsb_release -cs) main" | sudo tee /etc/apt/sources.list.d/redis.list
#sudo apt-get update
#sudo apt install redis -y
#pip install redis


# Neopixel
sudo apt install python-pip -y
sudo pip install rpi_ws281x adafruit-circuitpython-neopixel

# Chromium browser
sudo apt install chromium-browser -y

# TTS
python3 -m pip install gTTS
sudo apt install mpg123

> #### fix_monitor.sh

This script manages the X11 display synchronization and input peripheral alignment required for the robot's facial interface. It begins with a blocking polling loop that utilizes ``xrandr`` to monitor the system's video output ports, effectively pausing execution until a physical handshake is confirmed with the "HDMI-1" hardware. This ensures that subsequent configuration commands are not sent to a non-existent or uninitialized display buffer.

Once the display is active, the script executes an input coordinate mapping command using ``xinput``. This step is critical for synchronizing the Waveshare touch digitizer's coordinate space with the specific geometry of the HDMI output. By explicitly binding the "WaveShare WaveShare" input device to "HDMI-1," it prevents touch offset errors and ensures that user interactions on the screen accurately align with the graphical elements rendered by the web application.

In [ ]:
#! /bin/bash


export DISPLAY=:0


while true; do
    xrandr_output=$(xrandr)
    if [[ $xrandr_output == *"HDMI-1 connected"* ]]; then
        echo "HDMI-1 is connected!"
        break
    else
        echo "HDMI-1 not found. Waiting..."
        sleep 1
    fi
done

sleep 5
echo "Rotating display"
/usr/bin/xrandr --output HDMI-1 --rotate left

echo "Mapping input device to display"
/usr/bin/xinput map-to-output 8 "HDMI-1"

echo "Done"